In [1]:
import os
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    TimeDistributed, Conv2D, MaxPooling2D,
    Flatten, LSTM, Dense, Dropout
)
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


C:\Users\Harish\AppData\Roaming\Python\Python313\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [4]:
DATA_DIR = "data/train"
IMG_SIZE = (32, 32)
SEQUENCE_LENGTH = 10

class_names = sorted(os.listdir(DATA_DIR))
label_map = {name: idx for idx, name in enumerate(class_names)}

X, y = [], []

for label in class_names:
    folder = os.path.join(DATA_DIR, label)
    images = []

    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)
        img = load_img(img_path, target_size=IMG_SIZE)
        img = img_to_array(img) / 255.0
        images.append(img)

    # create sequences
    for i in range(len(images) - SEQUENCE_LENGTH):
        X.append(images[i:i+SEQUENCE_LENGTH])
        y.append(label_map[label])

X = np.array(X)
y = to_categorical(y, num_classes=2)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (50917, 10, 32, 32, 3)
y shape: (50917, 2)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y.argmax(axis=1)
)


In [ ]:
model = Sequential([
    TimeDistributed(Conv2D(32, (3,3), activation='relu'),
                    input_shape=(SEQUENCE_LENGTH, 32, 32, 3)),
    TimeDistributed(MaxPooling2D(2,2)),
    
    TimeDistributed(Conv2D(64, (3,3), activation='relu')),
    TimeDistributed(MaxPooling2D(2,2)),
    
    TimeDistributed(Flatten()),
    
    LSTM(64),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


C:\Users\Harish\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ time_distributed_5 (TimeDistributed) │ (None, 10, 30, 30, 32)      │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_6 (TimeDistributed) │ (None, 10, 15, 15, 32)      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_7 (TimeDistributed) │ (None, 10, 13, 13, 64)      │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_8 (TimeDistributed) │ (None, 10, 6, 6, 64)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed_9 (TimeDistributed) │ (None, 10, 2304)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │         606,464 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 2)                   │             130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 625,986 (2.39 MB)

 Trainable params: 625,986 (2.39 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=8
)


Epoch 1/10
5092/5092 ━━━━━━━━━━━━━━━━━━━━ 1003s 194ms/step - accuracy: 0.9660 - loss: 0.0917 - val_accuracy: 0.9877 - val_loss: 0.0364
Epoch 2/10
5092/5092 ━━━━━━━━━━━━━━━━━━━━ 953s 187ms/step - accuracy: 0.9919 - loss: 0.0243 - val_accuracy: 0.9918 - val_loss: 0.0205
Epoch 3/10
5092/5092 ━━━━━━━━━━━━━━━━━━━━ 1109s 218ms/step - accuracy: 0.9947 - loss: 0.0171 - val_accuracy: 0.9930 - val_loss: 0.0195
Epoch 4/10
5092/5092 ━━━━━━━━━━━━━━━━━━━━ 923s 178ms/step - accuracy: 0.9962 - loss: 0.0119 - val_accuracy: 0.9980 - val_loss: 0.0059
Epoch 5/10
4148/5092 ━━━━━━━━━━━━━━━━━━━━ 2:51 182ms/step - accuracy: 0.9964 - loss: 0.0101 


KeyboardInterrupt



In [8]:
model.save("drowsiness_lstm.h5")
print("✅ LSTM model saved")


✅ LSTM model saved
